In [2]:
import pandas as pd
import pulp

In [3]:
# Load the dataset
df = pd.read_csv("../data/processed/clean_supply_chain.csv")

print("Dataset Shape:", df.shape)
df.head()

Dataset Shape: (180519, 51)


,type,days_for_shipping_real,days_for_shipment_scheduled,benefit_per_order,sales_per_customer,delivery_status,late_delivery_risk,category_id,category_name,customer_city,...,order_state,order_status,product_card_id,product_category_id,product_image,product_name,product_price,product_status,shipping_date_dateorders,shipping_mode
0,DEBIT,3,4,91.25,314.64,Advance shipping,0,73,Sporting Goods,Caguas,...,Java Occidental,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-02-03 22:56:00,Standard Class
1,TRANSFER,5,4,-249.09,311.36,Late delivery,1,73,Sporting Goods,Caguas,...,Rajastán,PENDING,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-18 12:27:00,Standard Class
2,CASH,4,4,-247.78,309.72,Shipping on time,0,73,Sporting Goods,San Jose,...,Rajastán,CLOSED,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-17 12:06:00,Standard Class
3,DEBIT,3,4,22.86,304.81,Advance shipping,0,73,Sporting Goods,Los Angeles,...,Queensland,COMPLETE,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-16 11:45:00,Standard Class
4,PAYMENT,2,4,134.21,298.25,Advance shipping,0,73,Sporting Goods,Caguas,...,Queensland,PENDING_PAYMENT,1360,73,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2018-01-15 11:24:00,Standard Class


In [4]:
# Select required columns
optimization_df = df[[
    "order_id",
    "order_item_total",
    "days_for_shipping_real",
    "late_delivery_risk",
    "shipping_mode"
]].copy()

optimization_df.head()

,order_id,order_item_total,days_for_shipping_real,late_delivery_risk,shipping_mode
0,77202,314.64,3,0,Standard Class
1,75939,311.36,5,1,Standard Class
2,75938,309.72,4,0,Standard Class
3,75937,304.81,3,0,Standard Class
4,75936,298.25,2,0,Standard Class


In [5]:
from pulp import *

In [6]:
# Create optimization problem
model = LpProblem("Supply_Chain_Optimization", LpMinimize)

In [7]:
# Decision variables (1 = prioritize the order, 0 = don't prioritize)
decision_vars = {
    i: LpVariable(f"Order_{i}", cat="Binary")
    for i in optimization_df.index
}

print("Total decision variables:", len(decision_vars))

Total decision variables: 180519


In [8]:
# Use a smaller sample for testing
sample_df = optimization_df.sample(n=1000, random_state=42).reset_index(drop=True)

print("Sample Shape:", sample_df.shape)
sample_df.head()

Sample Shape: (1000, 5)


,order_id,order_item_total,days_for_shipping_real,late_delivery_risk,shipping_mode
0,31299,175.99,5,1,Standard Class
1,61023,245.00,2,1,First Class
2,111,244.90,2,0,Standard Class
3,50699,251.98,5,1,Standard Class
4,56606,107.97,2,0,Standard Class


In [9]:
from pulp import *

# Create optimization problem
model = LpProblem("Supply_Chain_Optimization", LpMinimize)

# Decision variables
decision_vars = {
    i: LpVariable(f"Order_{i}", cat="Binary")
    for i in sample_df.index
}

In [10]:
# Objective: Minimize total shipping days
model += lpSum(
    sample_df.loc[i, "days_for_shipping_real"] * decision_vars[i]
    for i in sample_df.index
)

In [11]:
model += lpSum(decision_vars[i] for i in sample_df.index) == 100

In [12]:
model.solve()

print("Status:", LpStatus[model.status])

Status: Optimal


In [13]:
# Get selected orders
selected_orders = sample_df[
    [decision_vars[i].value() == 1 for i in sample_df.index]
]

print("Total Selected Orders:", len(selected_orders))

selected_orders.head(10)

Total Selected Orders: 100


,order_id,order_item_total,days_for_shipping_real,late_delivery_risk,shipping_mode
1,61023,245.00,2,1,First Class
13,11381,233.62,2,0,Standard Class
15,29369,269.98,1,1,Same Day
16,23503,296.97,2,1,First Class
19,76021,11.31,2,0,Second Class
34,61778,181.99,1,1,Same Day
100,46588,89.99,2,1,First Class
102,13056,149.94,2,0,Standard Class
105,2441,47.25,2,0,Second Class
106,10956,248.98,2,0,Second Class


In [14]:
# Save optimized orders
selected_orders.to_csv("optimized_orders.csv", index=False)

print("Optimization results saved as optimized_orders.csv")

Optimization results saved as optimized_orders.csv
